In [ ]:
import pandas as pd

df = pd.read_csv("Dhaka_PM2.5_2022.csv")   # replace with your filename

print(df.head())
print(df.columns)

          Date (LT)  Hour  NowCast Conc.  Raw Conc. Conc. Unit  AQI  \
0  01/01/2022 02:00     2          130.1        129      ug/m3  190   
1  01/01/2022 03:00     3          128.1        126      ug/m3  188   
2  01/01/2022 04:00     4          125.1        121      ug/m3  187   
3  01/01/2022 05:00     5          125.6        126      ug/m3  187   
4  01/01/2022 06:00     6          124.2        120      ug/m3  186   

  AQI Category QC Name  
0    Unhealthy   Valid  
1    Unhealthy   Valid  
2    Unhealthy   Valid  
3    Unhealthy   Valid  
4    Unhealthy   Valid  
Index(['Date (LT)', 'Hour', 'NowCast Conc.', 'Raw Conc.', 'Conc. Unit', 'AQI',
       'AQI Category', 'QC Name'],
      dtype='object')


In [ ]:


# Convert to datetime
df["Date (LT)"] = pd.to_datetime(
    df["Date (LT)"],
    format="%d/%m/%Y %H:%M"
)

print("Number of rows:", len(df))
print("Start date:", df["Date (LT)"].min())
print("End date:", df["Date (LT)"].max())

Number of rows: 3531
Start date: 2022-01-01 02:00:00
End date: 2022-06-01 01:00:00


In [ ]:
#combine all AQI files
import pandas as pd
import glob
import os

# Folder containing the AQI CSVs
folder = "datasets"

# Select only the Dhaka PM2.5 yearly files
files = sorted(glob.glob(os.path.join(folder, "Dhaka_PM2.5_*.csv")))

print("Files found:")
for f in files:
    print(f)

# Read and combine
dfs = []
for file in files:
    df = pd.read_csv(file)
    dfs.append(df)

aqi = pd.concat(dfs, ignore_index=True)

print("\nTotal rows:", len(aqi))
print(aqi.head())

Files found:
datasets/Dhaka_PM2.5_2016.csv
datasets/Dhaka_PM2.5_2017.csv
datasets/Dhaka_PM2.5_2018.csv
datasets/Dhaka_PM2.5_2019.csv
datasets/Dhaka_PM2.5_2020.csv
datasets/Dhaka_PM2.5_2021.csv
datasets/Dhaka_PM2.5_2022.csv

Total rows: 53355
          Date (LT)  Hour  NowCast Conc.  Raw Conc. Conc. Unit  AQI  \
0  01/01/2016 01:00     1         -999.0       -999      ug/m3 -999   
1  01/01/2016 02:00     2         -999.0       -999      ug/m3 -999   
2  01/01/2016 03:00     3         -999.0       -999      ug/m3 -999   
3  01/01/2016 04:00     4         -999.0       -999      ug/m3 -999   
4  01/01/2016 05:00     5         -999.0       -999      ug/m3 -999   

  AQI Category  QC Name  
0          NaN  Missing  
1          NaN  Missing  
2          NaN  Missing  
3          NaN  Missing  
4          NaN  Missing  


In [ ]:
aqi["Date (LT)"] = pd.to_datetime(
    aqi["Date (LT)"],
    format="%d/%m/%Y %H:%M"
)

print("Rows:", len(aqi))
print("Start:", aqi["Date (LT)"].min())
print("End:", aqi["Date (LT)"].max())

Rows: 53355
Start: 2016-01-01 01:00:00
End: 2022-06-01 01:00:00


In [ ]:
weather = pd.read_csv("datasets/open-meteo.csv", skiprows=3)

weather["time"] = pd.to_datetime(weather["time"])

print(weather.shape)
print(weather["time"].min())
print(weather["time"].max())

(56256, 7)
2016-01-01 00:00:00
2022-06-01 23:00:00


In [ ]:
#Rename the time column:
weather = weather.rename(columns={"time": "Date (LT)"})

In [ ]:
#Merge:
merged = pd.merge(
    aqi,
    weather,
    on="Date (LT)",
    how="left"
)

In [ ]:
#verification of merge
print(merged.shape)

weather_cols = [
    "temperature_2m (°C)",
    "relative_humidity_2m (%)",
    "precipitation (mm)",
    "wind_speed_10m (km/h)",
    "vapour_pressure_deficit (kPa)",
    "cloud_cover (%)"
]

print(merged[weather_cols].isna().sum())

(53355, 14)
temperature_2m (°C)              0
relative_humidity_2m (%)         0
precipitation (mm)               0
wind_speed_10m (km/h)            0
vapour_pressure_deficit (kPa)    0
cloud_cover (%)                  0
dtype: int64


In [ ]:
#save the merged dataset
merged.to_csv("datasets/dhaka_aqi_weather_2016_2022.csv", index=False)

In [ ]:
import pandas as pd

full_weather = pd.read_csv("datasets/dhaka_weather_2016_2022.csv")

# Convert datetime column
full_weather["datetime"] = pd.to_datetime(full_weather["datetime"])

print(full_weather.shape)
print(full_weather["datetime"].min())
print(full_weather["datetime"].max())

(61368, 24)
2016-01-01 00:00:00
2022-12-31 23:00:00


In [ ]:
#merging this full weather
# Convert datetime
full_weather["datetime"] = pd.to_datetime(full_weather["datetime"])

# Rename so both dataframes have the same key
full_weather.rename(columns={"datetime": "Date (LT)"}, inplace=True)

# Merge
merged_full = pd.merge(
    aqi,
    full_weather,
    on="Date (LT)",
    how="left"
)

In [ ]:
#verify
print(merged_full.shape)

(53355, 31)


In [ ]:
weather_columns = [
    "temperature_2m",
    "relativehumidity_2m",
    "dewpoint_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "surface_pressure",
    "cloudcover",
    "cloudcover_low",
    "cloudcover_mid",
    "cloudcover_high",
    "windspeed_10m",
    "winddirection_10m",
    "windgusts_10m",
    "et0_fao_evapotranspiration",
    "vapor_pressure_deficit",
    "shortwave_radiation",
    "direct_radiation",
    "diffuse_radiation",
    "weathercode",
    "visibility",
    "soil_temperature_0cm",
    "soil_moisture_0_1cm"
]

print(merged_full[weather_columns].isna().sum())

temperature_2m                    0
relativehumidity_2m               0
dewpoint_2m                       0
apparent_temperature              0
precipitation                     0
rain                              0
surface_pressure                  0
cloudcover                        0
cloudcover_low                    0
cloudcover_mid                    0
cloudcover_high                   0
windspeed_10m                     0
winddirection_10m                 0
windgusts_10m                     0
et0_fao_evapotranspiration        0
vapor_pressure_deficit            0
shortwave_radiation               0
direct_radiation                  0
diffuse_radiation                 0
weathercode                       0
visibility                    53355
soil_temperature_0cm          53355
soil_moisture_0_1cm           53355
dtype: int64


In [ ]:
merged_full.to_csv(
    "datasets/merged_allcolumns_aqi_weather_2016_2022.csv",
    index=False
)